# DenseNet-121 - Individual Baseline Run


In [ ]:
%matplotlib inline
import sys
import os

import torch

sys.path.append(os.path.abspath("../src"))

from models import DenseNet121
from data import prepare_full_dataframe, prepare_data
from train import run_training_pipeline, run_smoke_test
from utils import (
    get_device,
    print_model_overrides,
)
import config

print(f"Python:         {sys.executable}")
print(f"PyTorch:        {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:            {torch.cuda.get_device_name(0)}")



In [ ]:
dataset_path = config.DATASET_PATH
print("Dataset location:", dataset_path)

metadata_file = os.path.join(dataset_path, "Data_Entry_2017.csv")
df = prepare_full_dataframe(metadata_file, dataset_path)

print(f"Total images:    {len(df)}")
print(f"Unique patients: {df['Patient ID'].nunique()}")

In [ ]:
print(df["split"].value_counts())

In [ ]:
train_loader, val_loader, test_loader = prepare_data(df, model_name="DenseNet121")
print("Train transforms:", train_loader.dataset.transform)
device = get_device()

## Smoke Test


In [ ]:
run_smoke_test(
    model_name="DenseNet121",
    model_builder=lambda: DenseNet121(num_classes=2, in_channels=1),
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    device=device,
    live_plot=False,
)

## Training


### Function Definitions

In [ ]:
run_results = []
baseline_auprc = 0.648

from notebook_experiment_runner import (
    cleanup_training_artifacts,
    print_planned_run_configuration,
    run_seed_experiment,
)

In [ ]:
# Optional per-run knobs for this notebook execution
seed_bank = tuple(config.TUNING_OVERRIDES.get("DenseNet121", {}).get("seed_bank", [16, 32, 64]))
epoch_log_file = "../outputs/experiment_outputs/densenet121_epoch_metrics.log"

def run_densenet_experiment(seeds=seed_bank, live_plot=True, reset_results=False, reset_epoch_log=True):
    run_seed_experiment(
        run_results=run_results,
        baseline_auprc=baseline_auprc,
        model_builder=lambda: DenseNet121(num_classes=2, in_channels=1),
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        device=device,
        model_name="DenseNet121",
        seeds=seeds,
        live_plot=live_plot,
        reset_results=reset_results,
        epoch_log_file=epoch_log_file,
        reset_epoch_log=reset_epoch_log,
    )

### Run

In [ ]:
print_planned_run_configuration(
    model_name="DenseNet121",
    model_builder=lambda: DenseNet121(num_classes=2, in_channels=1),
    device=device,
    resume_from_checkpoint=False,
)

In [ ]:
run_densenet_experiment()

## Complete Metrics

In [ ]:
import pandas as pd

summary_df = pd.DataFrame([
    {
        "Seed": r["seed"],
        "Best Epoch": r["best_epoch"],
        "Best Val AUPRC": f"{r['best_val_auprc']:.4f}" if r['best_val_auprc'] else "N/A",
        "Delta": f"{r['delta']:.4f}" if r['delta'] else "N/A",
    }
    for r in run_results
])

print("\n=== Run Comparison ===")
print(summary_df.to_string(index=False))

if len(run_results) > 1:
    valid_auprc = [r["best_val_auprc"] for r in run_results if r["best_val_auprc"] is not None]
    valid_best_epochs = [r["best_epoch"] for r in run_results if r["best_epoch"] is not None]
    mean_auprc = sum(valid_auprc) / len(valid_auprc)
    mean_delta = sum(r["delta"] for r in run_results if r["delta"] is not None) / len(valid_auprc)
    mean_best_epoch = sum(valid_best_epochs) / len(valid_best_epochs)
    std_auprc = (sum((x - mean_auprc) ** 2 for x in valid_auprc) / len(valid_auprc)) ** 0.5
    print(f"\nMean Best Epoch: {mean_best_epoch:.2f}")
    print(f"Mean Val AUPRC: {mean_auprc:.4f} ({mean_delta:+.4f})")
    print(f"Std Dev: {std_auprc:.4f}")
